In [1]:
import os
os.environ["HF_HUB_READ_TIMEOUT"] = "60"
os.environ["HF_HUB_CONNECT_TIMEOUT"] = "60"
from datasets import load_dataset

In [2]:
train_dataset = load_dataset('slegroux/tiny-imagenet-200-clean', split='train')                
valid_dataset = load_dataset('slegroux/tiny-imagenet-200-clean', split='validation')
test_dataset = load_dataset('slegroux/tiny-imagenet-200-clean', split='test')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/151M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/7.54M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/7.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/98179 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4909 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4923 [00:00<?, ? examples/s]

In [3]:
import torch
import torch.nn as nn
from collections import OrderedDict
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
device

device(type='cuda')

In [6]:
class SqueezeExcite(nn.Module):
    def __init__(self, in_channels, se_ratio=0.25):
        super().__init__()
        hidden = max(1, int(in_channels * se_ratio))
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, in_channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.se(x)

# -------------------------
# MBConv (user style)
# -------------------------
class MBConv(nn.Module):
    def __init__(self, expansion, num_input, num_output, stride, kernel_size, se_ratio=0.25):
        super().__init__()

        hidden_dim = expansion * num_input
        padding = kernel_size // 2

        self.use_expansion = (expansion != 1)
        self.use_residual = (stride == 1 and num_input == num_output)

        if self.use_expansion:
            self.conv1 = nn.Sequential(
                nn.Conv2d(num_input, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True),
            )
        else:
            self.conv1 = nn.Identity()

        self.conv2 = nn.Sequential(
            nn.Conv2d(
                hidden_dim if self.use_expansion else num_input,
                hidden_dim if self.use_expansion else num_input,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                groups=hidden_dim if self.use_expansion else num_input,
                bias=False,
            ),
            nn.BatchNorm2d(hidden_dim if self.use_expansion else num_input),
            nn.SiLU(inplace=True),
        )

        self.squeeze_excite = SqueezeExcite(hidden_dim if self.use_expansion else num_input, se_ratio)

        self.conv3 = nn.Sequential(
            nn.Conv2d(
                hidden_dim if self.use_expansion else num_input,
                num_output,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(num_output),
        )

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.squeeze_excite(out)
        out = self.conv3(out)

        if self.use_residual:
            out = out + x
        return out

# -------------------------
# EfficientNet-B0
# -------------------------
class EfficientNetB0(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True),
        )

        # Stages (from Table 1)
        self.stage2 = MBConv(1, 32, 16, stride=1, kernel_size=3)

        self.stage3 = nn.Sequential(
            MBConv(6, 16, 24, stride=2, kernel_size=3),
            MBConv(6, 24, 24, stride=1, kernel_size=3),
        )

        self.stage4 = nn.Sequential(
            MBConv(6, 24, 40, stride=2, kernel_size=5),
            MBConv(6, 40, 40, stride=1, kernel_size=5),
        )

        self.stage5 = nn.Sequential(
            MBConv(6, 40, 80, stride=2, kernel_size=3),
            MBConv(6, 80, 80, stride=1, kernel_size=3),
            MBConv(6, 80, 80, stride=1, kernel_size=3),
        )

        self.stage6 = nn.Sequential(
            MBConv(6, 80, 112, stride=1, kernel_size=5),
            MBConv(6, 112, 112, stride=1, kernel_size=5),
            MBConv(6, 112, 112, stride=1, kernel_size=5),
        )

        self.stage7 = nn.Sequential(
            MBConv(6, 112, 192, stride=2, kernel_size=5),
            MBConv(6, 192, 192, stride=1, kernel_size=5),
            MBConv(6, 192, 192, stride=1, kernel_size=5),
            MBConv(6, 192, 192, stride=1, kernel_size=5),
        )

        self.stage8 = MBConv(6, 192, 320, stride=1, kernel_size=3)

        # Head
        self.head = nn.Sequential(
            nn.Conv2d(320, 1280, 1, bias=False),
            nn.BatchNorm2d(1280),
            nn.SiLU(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.stage5(x)
        x = self.stage6(x)
        x = self.stage7(x)
        x = self.stage8(x)
        x = self.head(x)
        x = self.pool(x)
        x = x.flatten(1)
        x = self.classifier(x)
        return x

In [7]:
model = EfficientNetB0(200).to(device)

In [8]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def preprocess(example, transform):
    example["image"] = [transform(img) for img in example["image"]]
    return example

train_dataset = train_dataset.with_transform(lambda x: preprocess(x, transform))
valid_dataset = valid_dataset.with_transform(lambda x: preprocess(x, test_transform))
test_dataset = test_dataset.with_transform(lambda x: preprocess(x, test_transform))

In [9]:
optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.1,
            momentum=0.9,
            weight_decay=1e-4,
            nesterov=True
        )

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader   = DataLoader(valid_dataset, batch_size=64, shuffle=False, num_workers=4)
test_loader   = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[30, 60, 120],
    gamma=0.1
)
criterion = nn.CrossEntropyLoss()

In [10]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:
        images = batch['image'].to(device)
        label = batch['label'].to(device)

        # Forward
        output = model(images)
        loss = criterion(output, label)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        _, preds = output.max(1)
        correct += preds.eq(label).sum().item()
        total += label.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    for batch in dataloader:
        images = batch['image'].to(device)
        label = batch['label'].to(device)

        # Forward
        output = model(images)
        loss = criterion(output, label)
        running_loss += loss.item() * images.size(0)
        _, preds = output.max(1)
        correct += preds.eq(label).sum().item()
        total += label.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [11]:
best_val_acc = 0.0
num_epochs = 50
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc = validate(
        model, val_loader, criterion, device
    )
    
    scheduler.step() 
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )


Epoch [1/50] Train Loss: 4.3734, Train Acc: 0.0769 | Val Loss: 4.2844, Val Acc: 0.0821
Epoch [2/50] Train Loss: 3.6092, Train Acc: 0.1821 | Val Loss: 3.5383, Val Acc: 0.2000
Epoch [3/50] Train Loss: 3.1930, Train Acc: 0.2590 | Val Loss: 3.2438, Val Acc: 0.2506
Epoch [4/50] Train Loss: 2.8993, Train Acc: 0.3177 | Val Loss: 3.0389, Val Acc: 0.2942
Epoch [5/50] Train Loss: 2.6857, Train Acc: 0.3590 | Val Loss: 2.7453, Val Acc: 0.3516
Epoch [6/50] Train Loss: 2.5143, Train Acc: 0.3959 | Val Loss: 2.6164, Val Acc: 0.3775
Epoch [7/50] Train Loss: 2.3909, Train Acc: 0.4207 | Val Loss: 2.4730, Val Acc: 0.4062
Epoch [8/50] Train Loss: 2.2909, Train Acc: 0.4434 | Val Loss: 2.5535, Val Acc: 0.3862
Epoch [9/50] Train Loss: 2.2119, Train Acc: 0.4587 | Val Loss: 2.4637, Val Acc: 0.4184
Epoch [10/50] Train Loss: 2.1405, Train Acc: 0.4723 | Val Loss: 2.3651, Val Acc: 0.4363
Epoch [11/50] Train Loss: 2.0765, Train Acc: 0.4864 | Val Loss: 2.3621, Val Acc: 0.4376
Epoch [12/50] Train Loss: 2.0168, Train A

In [12]:
model_best = EfficientNetB0(200)
state_dict = torch.load("best_model.pth", map_location="cuda")
model.load_state_dict(state_dict)
# _, test_acc = validate(model, test_loader, criterion, device)
# print(test_acc)

<All keys matched successfully>

In [13]:
_, test_acc = validate(model, test_loader, criterion, device)
print(test_acc)

0.6288848263254113
